In [3]:
# %% [markdown]
# # Libraries and Env

# %%
import pandas as pd
import numpy as np
import os
from pathlib import Path
from dotenv import load_dotenv

# %%
def load_environment():
    """Load environment variables and return them as a dictionary."""
    load_dotenv(Path("../utils/.env"))  # Loads .env from project root
    env_vars = {
        "CLEANED_LANDCOVER_FOLDER": os.getenv("CLEANEDLANDCOVER_FOLDER"),
        "PREPROCESSED_LANDCOVER_FOLDER": os.getenv("PREPROCESSED_LANDCOVER_FOLDER"),
    }
    return env_vars

folders = load_environment()
cleaned_landcover_folder = folders["CLEANED_LANDCOVER_FOLDER"]
preprocessed_landcover_folder = folders["PREPROCESSED_LANDCOVER_FOLDER"]

# Ensure the output directory exists
Path(preprocessed_landcover_folder).mkdir(parents=True, exist_ok=True)

# Define file paths
CLEANED_IN_PATH = f"{cleaned_landcover_folder}/landcover_0p1_grid_cleaned.parquet"
PREPROCESSED_OUT_PATH = f"{preprocessed_landcover_folder}/preprocessed_landcover_grid.parquet"


# %% [markdown]
# # Read the Cleaned Grid File

# %%
# Retrieve the cleaned tabular dataset (Parquet)
try:
    land_cover_df = pd.read_parquet(CLEANED_IN_PATH)
    print(f"✅ Successfully read {len(land_cover_df)} rows.")
except FileNotFoundError:
    print(f"❌ Error: Cleaned file not found at {CLEANED_IN_PATH}.")
    raise

print("Initial columns:", land_cover_df.columns.tolist())
print(land_cover_df.head())


# %% [markdown]
# # Preprocessing Steps

# %% [markdown]
# ## 1. Handling Numerical Features: Area (Scaling/Transformation)

# %%
# Check if the 'area' column exists, as it might have been dropped in previous steps.
if 'area' in land_cover_df.columns:
    print("Area column summary:")
    print(land_cover_df["area"].describe())
    
    # As noted in your original analysis, the range is vast.
    # Log transformation is often preferred for highly skewed data over simple standardization
    # when the data must remain positive and is for machine learning models.
    
    # Add a small epsilon to avoid log(0) errors, although area should be > 0 here.
    epsilon = 1e-6 
    land_cover_df['area_log'] = np.log(land_cover_df['area'] + epsilon)
    
    # Drop the original highly skewed 'area' column
    land_cover_df = land_cover_df.drop(columns=['area'])
    print("Transformed 'area' to 'area_log' and dropped original 'area'.")
else:
    print("⚠️ 'area' column not found. Skipping numerical feature transformation.")


# %% [markdown]
# ## 2. Categorical Features: One-Hot Encoding `lcc_highlevel`

# %%
# Hot encoding lcc_highlevel into a vector of binary columns
if 'lcc_highlevel' in land_cover_df.columns:
    # Use get_dummies for one-hot encoding
    land_cover_encoded = pd.get_dummies(
        land_cover_df, 
        columns=['lcc_highlevel'], 
        prefix='lcc', 
        # drop_first=True is crucial to avoid multicollinearity (the 'Unclassified' or base category is represented by all zeros)
        drop_first=True 
    )
    print("Applied One-Hot Encoding to 'lcc_highlevel'.")
else:
    land_cover_encoded = land_cover_df.copy()
    print("⚠️ 'lcc_highlevel' column not found. Skipping one-hot encoding.")


# %% [markdown]
# ## 3. Missing Data Check (Imputation/Removal)

# %%
print("\nChecking for missing data:")
missing_data = land_cover_encoded.isnull().sum()
print(missing_data[missing_data > 0])

# Since this data is aggregated to a grid, any remaining NaNs (other than the optional 
# 'area' column which we handled) might indicate a cell with no valid land cover data.
# We'll stick to leaving them as is unless a specific imputation strategy is required.


# %% [markdown]
# ## 4. Final Cleanup (Dropping Redundant ID Columns)

# %%
land_cover_final = land_cover_encoded

# The final DataFrame should only contain 'cell_id' (for merging) and features.
# The 'id' column was likely dropped in the cleaning step, but we check again.
columns_to_drop_final = ['id']

for col in columns_to_drop_final:
    if col in land_cover_final.columns:
        land_cover_final = land_cover_final.drop(columns=col)
        print(f"Dropped final column: {col}")

# Note: 'cell_id' is kept as the primary key for merging with climate data.
# The original 'geometry' column is not present since we started from the tabular Parquet file.

print("\nFinal Preprocessed DataFrame Head:")
print(land_cover_final.head())
print("Final Columns:", land_cover_final.columns.tolist())


# %% [markdown]
# # Save the Preprocessed File

# %%
# Saving the final preprocessed DataFrame
try:
    # Save as Parquet, which is efficient and preserves data types.
    land_cover_final.to_parquet(PREPROCESSED_OUT_PATH)
    print(f"✅ Saved preprocessed land cover grid to:\n{PREPROCESSED_OUT_PATH}")
except Exception as e:
    # Fallback to CSV
    CSV_OUT_PATH = PREPROCESSED_OUT_PATH.replace(".parquet", ".csv")
    land_cover_final.to_csv(CSV_OUT_PATH, index=False)
    print(f"⚠️ Parquet failed (Error: {e}). Saved as CSV instead to:\n{CSV_OUT_PATH}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\

AttributeError: _ARRAY_API not found

✅ Successfully read 23322 rows.
Initial columns: ['cell_id', 'area', 'lcc_highlevel']
       cell_id          area lcc_highlevel
index                                     
0            0  3.358561e+11    Bare lands
1            1  3.358561e+11    Bare lands
2            2  3.358561e+11    Bare lands
3            3  3.358561e+11    Bare lands
4            4  3.358561e+11    Bare lands
Area column summary:
count    2.318900e+04
mean     2.460879e+11
std      2.772003e+11
min      1.447070e+05
25%      4.017569e+08
50%      1.043605e+11
75%      6.720004e+11
max      6.720004e+11
Name: area, dtype: float64
Transformed 'area' to 'area_log' and dropped original 'area'.
Applied One-Hot Encoding to 'lcc_highlevel'.

Checking for missing data:
area_log    133
dtype: int64

Final Preprocessed DataFrame Head:
       cell_id   area_log  lcc_Bare lands  lcc_Croplands  lcc_Forests  \
index                                                                   
0            0  26.539949            True  


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\venv\Lib\

AttributeError: _ARRAY_API not found

✅ Saved preprocessed land cover grid to:
../../PreprocessedDatasets/LandCoverDataset/preprocessed_landcover_grid.parquet
